In [ ]:
import re
from datasets import Dataset
from langchain_huggingface import HuggingFaceEmbeddings
import os
import dirtyjson
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
import ollama
from concurrent.futures import ThreadPoolExecutor

embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
storage_folder = os.path.join("../storage")
faiss_db = FAISS.load_local(
    folder_path=storage_folder, 
    embeddings=embedding_model, 
    allow_dangerous_deserialization=True)


def llm_response(query: str) -> str:
    context = get_context(query)

    response = ollama.chat(
        model="smollm2:latest",
        messages=[
                {
                "role": "system",
                "content": f"""
                    You are an assistant created to answer user questions based on the provided information.
                    You have access to the following inputs:

                    - **Query**: The specific question or instruction provided by the user.
                    - **Context**: Additionall contextual information that may help clarify or add details to the response: {context}

                    Your task is to use this information to provide accurate and clear answers to the user questions. When responding:

                    - Use the context to clarify or expand on the information in your response where applicable.
                    - Keep your answers concise, directly addressing the user query in a helpful manner.

                    Ensure that all responses are conversational and tailored to the user's specific needs.
                    """
                },
                {
                    "role": "user",
                    "content": query
                }
            ],
        options={
                "temperature": 0.0
            }
        )

    response = response["message"]["content"]
    return response

def multiquery(query: str) -> list[str]:
    response = ollama.chat(
        model="smollm2:latest",
        messages=[
            {
            "role": "system",
            "content": f"""
                You are an assistant tasked with generating query variations.

                Given a query provided by the user, follow these steps:

                1. Generate 3 related queries, focusing on the keywords of the original query.

                2. Ensure that each new query:
                    - Is unique and relevant, without repeating the original query.
                    - Matches the format of the original query:
                        - If the original query is a question, the 3 new queries must algo be questions.
                        - If the original query is an instruction, the 3 new queries must also be instructions.

                3. Output format:
                    - The result must be a readable JSON format using 'json.loads()' in Python, correctly formatted.

                Example of JSON output format:
                {{
                    "queries": ["query1", "query2", "query3"]
                }}

                Respond in JSON format.
                """
            },
            {
                "role": "user",
                "content": query
            }
        ],
        options={
            "temperature": 0.0
        }
    )

    queries_json = response["message"]["content"]
    queries = dirtyjson.loads(queries_json)
    queries_list = queries["queries"]

    return queries_list


def return_context(query: str) -> list[Document]:
    embedded_query = embedding_model.embed_query(query)
    
    context = faiss_db.similarity_search_by_vector(
        embedded_query,
        k=3
    )

    return context


def get_context(query: str) -> list[Document]:
    try:
        queries: list[str] = multiquery(query)
    except Exception as e:
        print(e)

    with ThreadPoolExecutor() as executor:
        results = list(executor.map(return_context, queries))

    full_context = []
    set_list = set()

    for sublist in results:
        for doc in sublist:
            if doc.page_content not in set_list:
                set_list.add(doc.page_content)
                full_context.append(doc)

    return full_context


questions = [
    "How much money does each player start with in Monopoly?",
    "What happens when a player lands on an unowned property?",
    "What is the rule when a player lands on an owned property without a house or hotel?",
    "What happens when a player lands on a Chance or Community Chest space?",
    "How many houses must be built on a property group before a hotel can be built?",
    "What happens if a player rolls doubles three times in a row?",
    "What is the purpose of the Jail space in Monopoly?",
    "How do you get out of Jail in Monopoly?",
    "What happens when a player passes Go?",
    "What is the rent on Boardwalk with a hotel in classic Monopoly?"
]

ground_truths = [
    "Each player starts with $1,500.",
    "The player may buy the property, or if they do not want it, it may be auctioned.",
    "The player must pay rent to the owner.",
    "The player draws the top card and follows the instructions on it.",
    "All properties in the color group must have four houses before a hotel can be built.",
    "The player is sent directly to Jail.",
    "Jail is a space where a player can be sent and may be held while still collecting rent and managing properties.",
    "A player can get out by rolling doubles, using a Get Out of Jail Free card, or paying a fine after three failed turns.",
    "The player collects $200.",
    "In classic Monopoly, Boardwalk with a hotel charges $2,000 in rent."
]


def tokenize(text: str) -> set[str]:
    """Tokenize the text"""
    return set(re.findall(r"\w+", text.lower()))


def token_f1(pred: str, ref: str) -> float:
    """
    Measure how similar two texts are using shared words.
    
    1. Tokenize both texts.
    2. Calculate how many tokens they have in common.
    3. Calculate precision and recall.
    4. Combine both in an F1 car.
    """
    pred_tokens = tokenize(pred)
    ref_tokens = tokenize(ref)

    if not pred_tokens or not ref_tokens:
        return 0.0

    common = len(pred_tokens & ref_tokens)
    precision = common / len(pred_tokens)
    recall = common / len(ref_tokens)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)


def cosine_sim(text_a: str, text_b: str) -> float:
    """
    Calculate semantic similarity between two texts using embeddings.

    1. Convert both texts into embeddings.
    2. Caculate dot product.
    3. Calculate the norm of each embedding.
    4. Apply COS formula.
    """
    emb_a = embedding_model.embed_query(text_a)
    emb_b = embedding_model.embed_query(text_b)

    dot = sum(a * b for a, b in zip(emb_a, emb_b))
    norm_a = sum(a * a for a in emb_a) ** 0.5
    norm_b = sum(b * b for b in emb_b) ** 0.5

    if norm_a == 0 or norm_b == 0:
        return 0.0

    return dot / (norm_a * norm_b)


def answer_correctness(answer: str, ground_truth: str) -> float:
    """Combine semantic similarity and similarity by words."""
    semantic = cosine_sim(answer, ground_truth)
    lexical = token_f1(answer, ground_truth)
    return 0.5 * semantic + 0.5 * lexical


def answer_relevancy(question: str, answer: str) -> float:
    """Measure how closely the answer relates to the question."""
    return cosine_sim(question, answer)


def context_precision(question: str, contexts: list[str], threshold: float = 0.0) -> float:
    """
    Measure how many of the retrieved chunks seem truly relevant to the question.
    
    1. Calculate similarity between the question and each context.
    2. Count how many scores exceed the threshold.
    3. Divide by the total.
    """
    if not contexts:
        return 0.0

    scores = [cosine_sim(question, context) for context in contexts]
    relevant = sum(1 for s in scores if s >= threshold)
    return relevant / len(scores)


def context_recall(ground_truth: str, contexts: list[str], threshold: float = 0.0) -> float:
    """
    Measure whether at least one of the retrieved chunks appears to cover the correct answer. 

    1. Compare ground_truth with each chunk.
    2. Obtain the best score.
    3. If the score exceed the threshold, return 1.0 else 0.0
    """
    if not contexts:
        return 0.0

    best_score = max(cosine_sim(ground_truth, context) for context in contexts)
    return 1.0 if best_score >= threshold else 0.0


def faithfulness(answer: str, contexts: list[str], threshold: float = 0.0) -> float:
    """
    Evaluate whether the model's response appears to be supported by the retrieved context.

    1. Divide the answer into sentences.
    2. For each sentence, find the most similar chunk.
    3. If the similarity exceed the threshold, it counts as supported.
    4. Returns the proportion of supported sentences. 
    """
    sentences = [s.strip() for s in re.split(r"[.!?]+", answer) if s.strip()]
    if not sentences or not contexts:
        return 0.0

    supported = 0
    for sentence in sentences:
        best_score = max(cosine_sim(sentence, context) for context in contexts)
        if best_score >= threshold:
            supported += 1

    return supported / len(sentences)


def collect_row(question: str, ground_truth: str, threshold: float) -> dict:
    context_docs = get_context(question)
    contexts = [doc.page_content for doc in context_docs]

    answer = llm_response(question)

    return {
        "question": question,
        "contexts": contexts,
        "answer": answer,
        "reference": ground_truth,
        "answer_correctness": answer_correctness(answer, ground_truth),
        "answer_relevancy": answer_relevancy(question, answer),
        "faithfulness": faithfulness(answer, contexts, threshold=threshold),
        "context_precision": context_precision(question, contexts, threshold=threshold),
        "context_recall": context_recall(ground_truth, contexts, threshold=threshold),
    }


def main():
    rows = []
    threshold = 0.5
    for question, ground_truth in zip(questions, ground_truths):
        row = collect_row(question, ground_truth, threshold)
        rows.append(row)

    evaluation_dataset = Dataset.from_list(rows)

    print(evaluation_dataset)
    print(evaluation_dataset.to_pandas()[[
        "question",
        "answer_correctness",
        "answer_relevancy",
        "faithfulness",
        "context_precision",
        "context_recall",
    ]])

    df = evaluation_dataset.to_pandas()

    df.to_excel("evaluation_threshold_00.xlsx", index=False)

    return evaluation_dataset


main()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8927.00it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Dataset({
    features: ['question', 'contexts', 'answer', 'reference', 'answer_correctness', 'answer_relevancy', 'faithfulness', 'context_precision', 'context_recall'],
    num_rows: 10
})
                                            question  answer_correctness  \
0  How much money does each player start with in ...            0.485617   
1  What happens when a player lands on an unowned...            0.403543   
2  What is the rule when a player lands on an own...            0.373666   
3  What happens when a player lands on a Chance o...            0.353491   
4  How many houses must be built on a property gr...            0.578922   
5  What happens if a player rolls doubles three t...            0.344342   
6  What is the purpose of the Jail space in Monop...            0.499373   
7            How do you get out of Jail in Monopoly?            0.537671   
8              What happens when a player passes Go?            0.155263   
9  What is the rent on Boardwalk with a hotel in .

Dataset({
    features: ['question', 'contexts', 'answer', 'reference', 'answer_correctness', 'answer_relevancy', 'faithfulness', 'context_precision', 'context_recall'],
    num_rows: 10
})